# Devstral and Voxtral on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Mistral's specialist models: Devstral 2 for agentic coding, and the Voxtral models,
which both cards list as taking speech as well as text. Whether an `input_audio` part
is accepted has moved between Regions and dates, so section 5 measures it and codes
against both answers. The general-purpose Mistral models and the size ladder are in
[`01-mistral-text-and-sizes`](01-mistral-text-and-sizes.ipynb).

| Model | `bedrock-mantle` ID | `bedrock-runtime` ID | Context window | Notes |
|---|---|---|---|---|
| [Devstral 2 123B](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-mistral-ai-devstral-2-123b.html) | `mistral.devstral-2-123b` | `mistral.devstral-2-123b` | 262,144 tokens | Agentic coding; Legacy, model EOL 30 March 2027 |
| [Voxtral Small 24B](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-mistral-ai-voxtral-small-24b-2507.html) | `mistral.voxtral-small-24b-2507` | `mistral.voxtral-small-24b-2507` | 32,768 tokens | Speech and text on the card; Legacy, model EOL 30 March 2027 |
| [Voxtral Mini 3B](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-mistral-ai-voxtral-mini-3b-2507.html) | `mistral.voxtral-mini-3b-2507` | `mistral.voxtral-mini-3b-2507` | 32,768 tokens | Smaller and faster; Legacy, model EOL 30 March 2027 |

IDs, Regions and modalities read from the three model cards on 20 October 2026. Each
model carries the same ID on `bedrock-runtime` and on `bedrock-mantle`, and every card
gives "Not supported" for both the Geo and the Global inference ID, so there is no
inference profile to look up and the bare ID above is what you send. Each card lists
sixteen Regions, every row In-Region with Geo and Global unticked. The Devstral card
ticks Text alone for input; both Voxtral cards tick Speech and Text and leave Audio
unticked, which is the split `GetFoundationModel` reported in `us-east-1` the same day
(`['TEXT']` for Devstral, `['SPEECH', 'TEXT']` for both Voxtral models). What the cards
tick is not a fixed answer about what the endpoints take: on 15 November 2026 an
`input_audio` part was accepted on both endpoints in each of the ten Regions where the
ID is reachable, 20 of 20 calls with a transcription, where on 25 October 2026 not one
of 64 attempts across the sixteen Regions the cards list was accepted. Section 5
measures the part and handles both answers (Things that differ).
The cards give a 256K context window for Devstral and 32K for either Voxtral, and tick
Standard, Priority and Flex among the service tiers while leaving Reserved unticked.

All three cards put these models in the **Legacy** stage, and the [lifecycle policy
for models launched before 7 September
2026](https://docs.aws.amazon.com/bedrock/latest/userguide/model-lifecycle-legacy.html)
gives all three the same dates: a Legacy date of 30 September 2026, a model EOL date
of 30 March 2027 and a public extended access start date still "To be announced"
(policy page read 18 October 2026; the cards carried the same stage and the same EOL
date when read again on 20 October 2026). The stage governs access and not only the
date: a new customer cannot start using a Legacy model, an existing account may lose
access after 15 days of inactivity, no new Provisioned Throughput can be created for
one, and after at least three months in the stage the model enters public extended
access, where the provider may raise the price. On, or soon after, the EOL date
requests fail. So read this notebook as a guide to models you may have to migrate off
rather than ones to start new work on, and check the policy page for the dates, which
move.

The control plane does not agree with the cards: in `us-east-1` on 18 October 2026, and
again on 20 October 2026, `GetFoundationModel` reported `modelLifecycle` status `ACTIVE`
for all three IDs, and all three answered 200 on both endpoints the same day, on 3 of 3
calls per model and endpoint in the second run. The `startOfLifeTime` the same field
carries does not match the cards' launch dates either: 17 February 2026 for Devstral
against the card's June 2025, and 2 December 2025 for both Voxtral models against 30
October 2025 and July 2025. Read the field as the access state the service reports for
your Region today, and the dated policy table as the announcement to plan against.

For the coding half this collection has a current alternative: `qwen.qwen3-coder-next`
is on both endpoints in `us-east-1`, its card reads `Model lifecycle: Active` with
`Model EOL date: N/A` (read 18 October 2026), and
[`04-qwen/02-qwen3-coder-and-vision`](../04-qwen/02-qwen3-coder-and-vision.ipynb)
covers it.

For the audio half these models do transcribe today, and they are Legacy, so the path
runs off them in time and there is no drop-in replacement on the routes this notebook
uses. On `bedrock-runtime` in `us-east-1` on 18 October 2026 the
catalogue lists speech input on four IDs besides these two:
`amazon.nova-2-sonic-v1:0` and `amazon.nova-2-5-sonic`, both reported `ACTIVE` and both
carrying `TEXT` among their output modalities, and the two TwelveLabs Marengo models,
whose output is `EMBEDDING`. The Nova Sonic models do transcribe: the [Nova 2 output
events page](https://docs.aws.amazon.com/nova/latest/nova2-userguide/sonic-output-events.html)
lists an "automatic speech recognition (ASR) transcription (what the user said)" among
the events the bidirectional stream returns (read 18 October 2026), so the catalogue
entry alone reads like a swap. They cannot be called the way this notebook calls Voxtral.
On the same day Converse refused both with `ValidationException: This action doesn't
support the model that you provided. Try again with a supported text or chat model.`,
Chat Completions on `bedrock-runtime`'s `/openai/v1` refused both with 400 `The provided
model ID is not supported. Try again with valid model ID.`, and `bedrock-mantle`
returned 404 `The model '…' does not exist`. [API compatibility by
models](https://docs.aws.amazon.com/bedrock/latest/userguide/models-api-compatibility.html)
gives `InvokeModelWithBidirectionalStream` as Nova 2 Sonic's sole invocation path, and
says that API "cannot be used with Amazon Bedrock API keys and requires standard AWS
credential-based authentication" (read 18 October 2026), where this notebook
authenticates with a Bedrock API key throughout. So moving off Voxtral for transcription
means a different API, a different way to authenticate and a transcript read from an
event stream rather than from `choices[0].message.content`. No other Mistral model on
either catalogue takes speech in for that Region and day, so plan the API change rather
than waiting for a named successor. Section 5 transcribes the clip on Bedrock's Chat
Completions route and handles the refusal the part returns elsewhere, each with the
Region and the date it was measured on.

Devstral's window is eight times either Voxtral's, and the window holds the prompt and
the answer together, so `max_tokens` has to leave room for the input. With a five-token
prompt in `us-east-1` in October 2026, Devstral took `max_tokens=262139` and refused
262140, and both Voxtral models took 32763 and refused 32764.

Neither Voxtral card publishes a `Max output tokens` figure. Devstral's publishes 32K,
and the endpoint does not enforce it: Devstral accepted `max_tokens=262139`, eight times
the documented figure, in `us-east-1` in October 2026. Treat the window minus your prompt
as the bound you can measure, and Devstral's 32K as a figure that may start being
enforced.

On `bedrock-mantle` these models serve the OpenAI-compatible **Chat Completions** API at
the bare `/v1` path, which this notebook uses. On `bedrock-runtime` they serve
**Converse** and **InvokeModel** under the same IDs: section 6 calls InvokeModel on
Devstral, and the Converse request shape is in
[`01-mistral-text-and-sizes`](01-mistral-text-and-sizes.ipynb) section 9, where the same
endpoint serves the general-purpose models. Each of the three cards ticks Chat Completions,
Invoke and Converse among the APIs supported, marks Responses not supported, and ticks
both endpoints (read 2 November 2026). Converse answered `end_turn` with a single `text`
block on the three IDs in `us-east-1` on 2 November 2026.

**Covered:** code generation, checking generated code without running it, a coding agent
with tools, a structured code review, a Voxtral transcription beside the refusal its audio
part returns elsewhere, and InvokeModel on `bedrock-runtime`.

## Setup

Install the pinned requirements first (`pip install -r ../requirements.txt`, Python 3.11
or later). The notebook calls Bedrock with the OpenAI SDK. From `_shared/bedrock.py`,
this repository's own helper module, it takes a lenient JSON parser, static checks for
generated code, and the sample audio clip.

Endpoints, authentication and URL paths are explained in
[`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb).

In [1]:
import ast
import json
import sys

import boto3
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

sys.path.insert(0, "../_shared")
from bedrock import parse_json_lenient

REGION = "us-east-1"
DEVSTRAL = "mistral.devstral-2-123b"
VOXTRAL_SMALL = "mistral.voxtral-small-24b-2507"

# A short-term Bedrock API key minted from your AWS credentials. Mint it when you
# build the client, because the key expires.
client = OpenAI(
    base_url=f"https://bedrock-mantle.{REGION}.api.aws/v1",
    api_key=provide_token(region=REGION),
    timeout=180,
)

# bedrock-runtime: boto3 signs each request with SigV4, so there is no key to mint.
runtime = boto3.client("bedrock-runtime", region_name=REGION)

## 1. Code generation with Devstral

Ask for a function to a specification. A low `temperature` keeps the output close to the
model's most likely answer, which suits code; this family accepts `temperature` and
`top_p`.

In [2]:
SPEC = """Write a Python function `retry(fn, attempts=3, base_delay=0.1)` that:
- calls fn() and returns its result
- retries on any exception, up to `attempts` calls in total
- sleeps base_delay * (2 ** attempt) between attempts
- re-raises the last exception if every attempt fails
Include a short docstring. Reply with one Python code block and nothing else."""

response = client.chat.completions.create(
    model=DEVSTRAL,
    messages=[{"role": "user", "content": SPEC}],
    max_tokens=1000,
    temperature=0.2,
)
generated = response.choices[0].message.content or ""
print(generated.strip()[:1200])
print(f"\nfinish_reason={response.choices[0].finish_reason}")

```python
import time

def retry(fn, attempts=3, base_delay=0.1):
    """
    Retries calling fn() up to `attempts` times with exponential backoff.

    Args:
        fn: The function to call.
        attempts: Maximum number of attempts (default: 3).
        base_delay: Initial delay in seconds (default: 0.1).

    Returns:
        The result of fn() if successful.

    Raises:
        The last exception if all attempts fail.
    """
    last_exception = None
    for attempt in range(attempts):
        try:
            return fn()
        except Exception as e:
            last_exception = e
            if attempt < attempts - 1:
                time.sleep(base_delay * (2 ** attempt))
    raise last_exception
```

finish_reason=stop


## 2. Checking the code without running it

Model output is untrusted input, and this kernel holds your AWS credentials, so do not
`exec()` it here. `inspect_code` and `check_spec` from `_shared/bedrock.py` parse the
source with `ast` and report what it declares, without running anything. The
specification asks for a backoff, so the cell also checks that the code calls `sleep`.
To run generated code, use a sandbox with no credentials and no network access, such as
Amazon Bedrock AgentCore Code Interpreter.

In [3]:
from bedrock import check_spec, extract_code_block, inspect_code

source = extract_code_block(generated)
info = inspect_code(source)
verdict = check_spec(source, function="retry", params=["fn", "attempts", "base_delay"])
sleeps = any(name.endswith("sleep") for name in info["calls"])

print("parses   :", info["parses"], info["error"] or "")
print("functions:", info["functions"])
print("calls    :", sorted(set(info["calls"])))
print(f"\nsignature matches: {verdict['ok']}   backs off with sleep: {sleeps}")
print("meets the spec" if verdict["ok"] and sleeps else f"does not meet the spec: {verdict['reason'] or 'no sleep call'}")

parses   : True 
functions: {'retry': ['fn', 'attempts', 'base_delay']}
calls    : ['fn', 'range', 'sleep']

signature matches: True   backs off with sleep: True
meets the spec


## 3. A coding agent with tools

Devstral is built for working through a codebase with tools. The workspace here is a
dictionary with two files, and the task is to make `fetch()` honour a retry setting.
`check_retries` checks the result with `ast`, so the model's code is never run. The loop
validates each call against the tool's parameters and stops after eight rounds.

In [4]:
WORKSPACE = {
    "config.py": "TIMEOUT = 30\nRETRIES = 3\n",
    "client.py": ("import config\n\n\ndef fetch(url):\n    # TODO: honour config.RETRIES\n"
                  "    return _get(url, timeout=config.TIMEOUT)\n\n\n"
                  "def _get(url, timeout):\n    return {'url': url, 'timeout': timeout}\n"),
}


def list_files() -> dict:
    return {"files": sorted(WORKSPACE)}


def read_file(path: str) -> dict:
    return {"path": path, "content": WORKSPACE.get(path), "exists": path in WORKSPACE}


def write_file(path: str, content: str) -> dict:
    WORKSPACE[path] = content
    return {"path": path, "bytes": len(content)}


def check_retries(path: str) -> dict:
    """Pass when fetch() loops and reads config.RETRIES, checked with ast. Never runs it."""
    try:
        tree = ast.parse(WORKSPACE.get(path, ""))
    except SyntaxError as exc:
        return {"ok": False, "reason": f"syntax error: {exc.msg}"}
    fetch = next((n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == "fetch"), None)
    if fetch is None:
        return {"ok": False, "reason": "no fetch() in " + path}
    loops = any(isinstance(n, (ast.For, ast.While)) for n in ast.walk(fetch))
    reads = any(isinstance(n, ast.Attribute) and n.attr == "RETRIES" for n in ast.walk(fetch))
    return {"ok": loops and reads, "loops": loops, "reads_RETRIES": reads}

In [5]:
def tool(name: str, description: str, *params: str) -> dict:
    schema = {"type": "object", "properties": {p: {"type": "string"} for p in params}, "required": list(params)}
    return {"type": "function", "function": {"name": name, "description": description, "parameters": schema}}


CODE_TOOLS = [
    tool("list_files", "List the files in the workspace."),
    tool("read_file", "Read a file from the workspace.", "path"),
    tool("write_file", "Write a file to the workspace.", "path", "content"),
    tool("check_retries", "Check whether fetch() in a file retries.", "path"),
]
DISPATCH = {"list_files": list_files, "read_file": read_file, "write_file": write_file,
            "check_retries": check_retries}


def run_tool(name: str, args: dict) -> dict:
    """Refuse a call that names an unknown tool or does not match the tool's parameters."""
    spec = next((t["function"] for t in CODE_TOOLS if t["function"]["name"] == name), None)
    if spec is None:
        return {"error": f"unknown tool {name}"}
    params = spec["parameters"]
    missing = [k for k in params["required"] if k not in args]
    unknown = [k for k in args if k not in params["properties"]]
    wrong_type = [k for k in args if k in params["properties"] and params["properties"][k].get("type") == "string"
                  and not isinstance(args[k], str)]
    if missing or unknown or wrong_type:
        return {"error": f"bad arguments: missing {missing}, unknown {unknown}, not a string {wrong_type}"}
    return DISPATCH[name](**args)

In [6]:
TASK = ("client.py has a TODO: fetch() ignores config.RETRIES. Rewrite fetch() so it retries up "
        "to config.RETRIES times, write the file back, then run check_retries on it.")

messages = [{"role": "user", "content": TASK}]
for round_no in range(1, 9):  # bounded, so an agent that cannot finish stops
    response = client.chat.completions.create(
        model=DEVSTRAL, messages=messages, tools=CODE_TOOLS, max_tokens=2000, temperature=0.2
    )
    message = response.choices[0].message
    if not message.tool_calls:
        break
    messages.append(message.model_dump(exclude_none=True))
    for call in message.tool_calls:
        args = parse_json_lenient(call.function.arguments or "{}")
        result = run_tool(call.function.name, args)
        print(f"round {round_no}: {call.function.name}({', '.join(args)}) -> {json.dumps(result)[:80]}")
        messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})
else:
    raise RuntimeError("the agent did not finish in 8 rounds")

# Print the whole reply. A slice cuts a longer summary mid-sentence with nothing in the
# output to show that it was cut.
print("\nanswer:", (message.content or "").strip())
print("\nclient.py now:\n" + WORKSPACE["client.py"])
# The loop breaks on any response without tool_calls, a reply the 2,000-token cap ended
# included, so print finish_reason beside the check: the workspace can be correct while
# the summary of it stops mid-sentence.
print("check:", check_retries("client.py"),
      " finish_reason:", response.choices[0].finish_reason)

round 1: read_file(path) -> {"path": "client.py", "content": "import config\n\n\ndef fetch(url):\n    # TODO


round 2: write_file(path, content) -> {"path": "client.py", "bytes": 357}


round 3: check_retries(path) -> {"ok": true, "loops": true, "reads_RETRIES": true}



answer: The `fetch()` function in `client.py` has been successfully updated to retry up to `config.RETRIES` times. The changes have been verified, and the function now correctly implements retry logic.

client.py now:
import config
import time


def fetch(url):
    retries = config.RETRIES
    for attempt in range(retries):
        try:
            return _get(url, timeout=config.TIMEOUT)
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(1)


def _get(url, timeout):
    return {'url': url, 'timeout': timeout}

check: {'ok': True, 'loops': True, 'reads_RETRIES': True}  finish_reason: stop


## 4. A structured code review

Ask for review findings as JSON, so a pipeline can act on them: where each problem is,
what it is, and how severe. The code builds an SQL query from user input, so the cell
checks that the review flagged SQL injection in `build_query`.

In [7]:
SUSPECT = '''
import os


def load_secret(name):
    return os.environ[name]


def build_query(table, user_input):
    return f"SELECT * FROM {table} WHERE name = '{user_input}'"


def divide_all(values, divisor):
    return [v / divisor for v in values]
'''
FINDING = {
    "type": "object",
    "properties": {
        "location": {"type": "string"},
        "issue": {"type": "string"},
        "severity": {"type": "string", "enum": ["low", "medium", "high"]},
    },
    "required": ["location", "issue", "severity"],
    "additionalProperties": False,
}
REVIEW_SCHEMA = {
    "type": "object",
    "properties": {"findings": {"type": "array", "items": FINDING},
                   "overall": {"type": "string", "enum": ["approve", "request_changes"]}},
    "required": ["findings", "overall"],
    "additionalProperties": False,
}

In [8]:
response = client.chat.completions.create(
    model=DEVSTRAL,
    messages=[{"role": "user", "content": f"Review this code for bugs and security issues.\n```python\n{SUSPECT}```"}],
    response_format={"type": "json_schema", "json_schema": {"name": "review", "strict": True, "schema": REVIEW_SCHEMA}},
    max_tokens=2000,
    temperature=0.2,
)
choice = response.choices[0]
if choice.finish_reason != "stop":
    raise RuntimeError(f"incomplete output (finish_reason={choice.finish_reason}); raise max_tokens")

review = parse_json_lenient(choice.message.content)
print("overall:", review["overall"])
for finding in review["findings"]:
    print(f"[{finding['severity']:6}] {finding['location']}: {finding['issue'][:80]}")
flagged = any("build_query" in f["location"] and "sql" in f["issue"].lower() for f in review["findings"])
print(f"\nflagged SQL injection in build_query: {flagged}")

overall: request_changes
[medium] load_secret: Missing error handling
[high  ] build_query: SQL injection vulnerability
[medium] divide_all: Division by zero risk

flagged SQL injection in build_query: True


## 5. Voxtral on speech, and the refusal to code against

Both Voxtral cards tick Speech for input, and whether the endpoint takes an
`input_audio` part has moved between measurements. On 15 November 2026 the part was
accepted in the ten Regions, of the sixteen the cards list, where the ID is reachable,
on `bedrock-mantle`'s `/v1` and on `bedrock-runtime`'s `/openai/v1`: 20 of 20 calls
answered 200 with
`finish_reason=stop` and a transcription of the clip, in 0.4 to 1.8 seconds. On 25
October 2026, across the same sixteen Regions, not one of 64 attempts was accepted, and
`eu-west-2` and `ap-northeast-1` transcribed the same clip on 21 October 2026 and
refused it four days later.

So measure the part in the Region and account you plan to use before you depend on it,
and handle the 400 either way: on 15 November 2026
`ap-southeast-3` still returned 400 `Model does not support audio modality` on
`bedrock-mantle`, and the six Regions whose runtime catalogue does not carry the IDs
returned 400 `The provided model identifier is invalid.` from `bedrock-runtime`
instead. Five of those six sent nothing back on `bedrock-mantle` rather than refusing,
`ap-southeast-4`, `ap-northeast-2`, `ap-southeast-1`, `eu-central-1` and `eu-north-1`,
measured with a 60-second client timeout and no retries. Pointing `REGION` at one of
those five, or at `ap-southeast-3`, raises a timeout rather than an answer, and it takes
three times the timeout to do it, because the setup cell's client keeps the SDK default
`max_retries=2`: with `REGION = "ap-southeast-3"` the cell printed the 400 for the audio
part and then raised `openai.APITimeoutError` after 544.8 seconds on the text-only call,
which is `timeout=180` three times over. In each of the ten reachable Regions both halves
of the cell answered, and in `ap-southeast-3` neither did, so a refusal and a working
text-only control did not appear together in the Regions measured on 15 November 2026:
on a run that prints the refusal branch, the text control below it is the call that goes
silent.

The mantle catalogue reaches wider than the cards do for these IDs, and a catalogue entry
is not a promise of an answer. `GET /v1/models` on `bedrock-mantle` in `ca-central-1`,
which is not one of the sixteen Regions either card lists, carries both Voxtral IDs,
and on 16 November 2026 the audio call to `mistral.voxtral-small-24b-2507` there sent
nothing back within a 30-second client timeout, while `bedrock-runtime` in that Region
returned 400 `The provided model identifier is invalid.` The sibling notebook makes the
same point for Mistral Large 3, whose card lists seven Regions against the catalogue's
thirteen ([`01-mistral-text-and-sizes`](01-mistral-text-and-sizes.ipynb)).

The cell below sends the audio part first, in `us-east-1`, then asks the same model a
question whose answer can be checked, and it catches the status error a refusal raises
so that both paths print. The clip is a seven-second excerpt that ends mid-sentence, so
a transcript that ends mid-sentence is the clip rather than a capped answer; read
`finish_reason` to tell the two apart. The transcript is checked against a word from the
clip rather than on the status alone, because these models mis-hear words: over 13
calls to Voxtral Small on `bedrock-mantle` in `us-east-1` on 15 November 2026 every
transcript named S3,
while one of six Voxtral Mini transcripts rendered the tiering mechanism as a
"sub-structuring mechanism" and dropped the name, and one of the 13 opened with a
sentence that is not in the clip. The capital question is the model's own answer rather
than a reachability test: on 26 October 2026 in `us-east-1`, one of 21 calls answered
`Sydney` and the other 20 answered `Canberra`, each with `finish_reason=stop`, so
`capital correct` can read `False` on a run where the 200 still arrived.

These models are Legacy, so read the paragraph at the top of this notebook on moving off
them before you build on the transcription. Neither primary source says where the part
is accepted: both cards tick Speech for input and tick In-Region for each of their
sixteen Regions, and `GetFoundationModel` reported `['SPEECH', 'TEXT']` for both models
in `us-east-1` on 20 October 2026.


In [9]:
from openai import APIStatusError

from bedrock import clip_mp3_b64

# Whether the endpoint takes an input_audio part has moved between measurements, so the
# cell codes against both answers rather than one: a transcription, or the 400.
try:
    audio_response = client.chat.completions.create(
        model=VOXTRAL_SMALL,
        messages=[{"role": "user", "content": [
            {"type": "input_audio",
             "input_audio": {"data": clip_mp3_b64(), "format": "mp3"}},
            {"type": "text", "text": "Transcribe this audio verbatim."},
        ]}],
        max_tokens=300,
    )
    choice = audio_response.choices[0]
    transcript = (choice.message.content or "").strip()
    # The clip says "You can save costs by leveraging the tiering mechanism in Amazon
    # S3", so check a word from it rather than the status alone.
    print("transcript  :", transcript)
    print("mentions S3 :", "s3" in transcript.lower(),
          " finish_reason:", choice.finish_reason)
except APIStatusError as exc:
    # On this mantle client a refusal reads `Model does not support audio modality`.
    # The same call to bedrock-runtime's /openai/v1 reads `The provided model
    # identifier is invalid.` where that catalogue has no entry for the ID, which on
    # mantle times out instead. str() carries the status and the body.
    print("audio refused:", exc)

# The same ID, client and Region answer a text-only request, which separates a refused
# modality from a model or a credential you cannot reach.
response = client.chat.completions.create(
    model=VOXTRAL_SMALL,
    messages=[{"role": "user",
               "content": "What is the capital of Australia? Answer in one word."}],
    max_tokens=50,
)
answer = (response.choices[0].message.content or "").strip()
print("text answer :", answer)
print("capital correct:", "canberra" in answer.lower(),
      " finish_reason:", response.choices[0].finish_reason)

transcript  : You can save cost by leveraging the tiering mechanism in Amazon S3. In Amazon S3, you
mentions S3 : True  finish_reason: stop
text answer : Canberra
capital correct: True  finish_reason: stop


## 6. InvokeModel on `bedrock-runtime`

The Sample Code on each of the three cards carries an Invoke tab calling `invoke_model`
with the bare ID, beside the Chat Completions tab (read 2 November 2026), so a reader who
starts from the card may well arrive at this API before the calls above. The body is the
Chat Completions shape rather than the `{"prompt": "<s>[INST] … [/INST]"}` older Mistral
samples send (Things that differ), and the reply is the Chat Completions envelope: the
answer under `choices[0].message`, with `refusal` beside `content` and `role`, and
`finish_reason` where Converse gives `stopReason`.

Over two calls on each of the three IDs in `us-east-1` on 2 November 2026, 6 of 6 came
back `finish_reason` `stop` at 3 or 4 completion tokens, and 5 of the 6 replies named
Canberra: Voxtral Mini answered `Sydney, Melbourne` on one of its two, which is the size
of the model rather than the API, so check the answer you depend on rather than the
status alone. The general-purpose models take the same body on this path
([`01-mistral-text-and-sizes`](01-mistral-text-and-sizes.ipynb) section 10).


In [10]:
# InvokeModel carries the Chat Completions body as bytes and returns the same envelope;
# a `prompt` body with the old [INST] markers is refused (Things that differ).
response = runtime.invoke_model(
    modelId=DEVSTRAL,
    body=json.dumps({
        "messages": [{"role": "user", "content": "What is the capital of Australia? "
                                                 "Answer in one word."}],
        "max_tokens": 200,
    }),
)
payload = json.loads(response["body"].read())
choice = payload["choices"][0]
# Check the status before the text: a reply the budget cut off still parses.
if choice["finish_reason"] != "stop":
    raise RuntimeError(f"incomplete reply (finish_reason={choice['finish_reason']}); "
                       "raise max_tokens")
answer = (choice["message"].get("content") or "").strip()
print("message keys :", sorted(choice["message"]))
print("answer       :", answer, "| Canberra:", "canberra" in answer.lower())
print("finish_reason:", choice["finish_reason"],
      "| completion_tokens:", payload["usage"]["completion_tokens"])


message keys : ['content', 'refusal', 'role']
answer       : Canberra | Canberra: True
finish_reason: stop | completion_tokens: 3


## Things that differ

What trips people up on these models, and what to do about it.

| Behaviour | What you see | What to do |
|---|---|---|
| Lifecycle state | All three cards mark these models `Legacy` with a model EOL date of 30 March 2027, while `GetFoundationModel` reports `modelLifecycle` status `ACTIVE` for all three in `us-east-1` (18 and 20 October 2026) | Plan against the card's EOL date rather than the field, and read `modelLifecycle` in your own account and Region before you build on any of them. For agentic coding, `qwen.qwen3-coder-next` is Active ([`04-qwen/02-qwen3-coder-and-vision`](../04-qwen/02-qwen3-coder-and-vision.ipynb)); for transcription these models still answer, and there is no drop-in replacement on the routes used here once they go, for the reasons at the top of this notebook. |
| Responses API | Refused on both routes, and the message names the route, so the text differs by where you called. `POST /v1/responses` on `bedrock-mantle` returns 400 `The model 'mistral.voxtral-small-24b-2507' does not support the '/v1/responses' API`, and `POST /openai/v1/responses` on `bedrock-runtime` returns 400 `The model 'mistral.voxtral-small-24b-2507' does not support the '/openai/v1/responses' API`. Both bodies carry `type: invalid_request_error` and `code: validation_error`, and all three models are refused the same way, each message naming the ID it was called with (`us-east-1`, 18 and 20 October 2026; 3 of 3 calls per model and route on the second run) | Use Chat Completions on either endpoint, or Converse on `bedrock-runtime`: all three models answered 200 on `bedrock-mantle`'s `/v1`, on `bedrock-runtime`'s `/openai/v1` and through Converse on the same day. All three cards mark Responses unsupported in their *APIs supported* table, and the [Voxtral Small card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-mistral-ai-voxtral-small-24b-2507.html)'s Sample Code section calls `client.responses.create` against `bedrock-runtime`'s `/openai/v1` regardless, while the Devstral and Voxtral Mini cards' Sample Code shows Chat Completions, Invoke and Converse only (read 20 October 2026), so code against the refusal rather than the sample. |
| Path prefix | On `bedrock-mantle`, `/openai/v1/chat/completions` returns 400 ``model `…` isn't supported on this route`` for these models. On `bedrock-runtime` that path answers 200 for all three models under their bare IDs (`us-east-1`, 18 October 2026) | Use the bare `/v1` path on `bedrock-mantle`. The prefix is a property of the model rather than of the endpoint: most models on `bedrock-mantle` are served under the bare `/v1`, these among them, and some are served under `/openai/v1` instead, so the 400 names a model-and-route pair. `google.gemma-4-e2b` answers 200 on mantle's `/openai/v1` and returns the same 400 on its bare `/v1` (17 October 2026, `us-east-1`). Take the base URL from the Programmatic Access table on the model's card. `bedrock-runtime` has an `/openai/v1` route of its own, which also serves Devstral and Voxtral. |
| Legacy `prompt` body on InvokeModel | ``ValidationException: Failed to deserialize the JSON body into the target type: missing field `messages` at line 1 column 103`` for `{"prompt": "<s>[INST] Say OK. [/INST]", "max_tokens": 20}` on Devstral 2, and the same message on Voxtral Mini and on the general-purpose models (`us-east-1`, 2 November 2026). The column moves with the length of the model ID, 103, 108 and 111 for IDs of 23, 28 and 31 characters against the same 57-byte body, so it indexes into a body the service assembled rather than the one you sent | Send the Chat Completions shape, `messages` rather than `prompt`, which is what the Invoke tab on each card shows (section 6). The older `[INST]` prompt format belongs to the earlier Mistral models on Bedrock and is not what these IDs take. |
| Empty answer | HTTP 200, `content` is `None`, `finish_reason` is `length` | The model spent `max_tokens` before answering, often on reasoning. Raise the budget, and keep it below the window minus your prompt: the windows are 262,144 tokens on Devstral and 32,768 on either Voxtral. Check `finish_reason` before using the text. |
| Output budget | Two messages from the model, by how far over. Past the window minus the prompt, 400 `This model's maximum context length is 32768 tokens. However, you requested 32764 output tokens and your prompt contains at least 5 input tokens, for a total of at least 32769 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=5)`, whose trailing `value` is the prompt's token count. Well past the window, 400 `'max_tokens' (10000000) exceeds model maximum (32768)`, which names it. On `bedrock-runtime` a third message comes from Bedrock rather than from the model, `ValidationException: The maximum tokens you requested exceeds the model limit of 32768. Try again with a maximum tokens value that is lower than 32768.` (October 2026) | Keep `max_tokens` below the window minus your prompt. The figure a message names is the window, rather than a `max_tokens` you can set. |
| Audio input | Whether an `input_audio` part is taken has moved between measurements, so code against both answers. On 15 November 2026 it was accepted for Voxtral Small on `bedrock-mantle`'s `/v1` and on `bedrock-runtime`'s `/openai/v1` in each of the ten Regions, of the sixteen the cards list, where the ID is reachable, 20 of 20 calls answering 200 with `finish_reason=stop` and a transcription, in 0.4 to 1.8 seconds, and 12 of 12 calls across both models and both endpoints in `us-east-1` did the same. On 25 October 2026, across the same sixteen Regions, not one of 64 attempts was accepted: 46 returned 400 `Model does not support audio modality`, 12 returned 400 `The provided model identifier is invalid.` from `bedrock-runtime` in the six Regions whose runtime catalogue does not carry the IDs, and 6 timed out on `bedrock-mantle`. `eu-west-2` and `ap-northeast-1` transcribed the clip on 21 October 2026 and refused it on 25 October. Both refusals still exist: on 15 November 2026 `ap-southeast-3` returned `Model does not support audio modality` on mantle, and the six runtime-absent Regions returned `The provided model identifier is invalid.` | Measure the part in your own Region and account before you build on it, and catch `APIStatusError` either way: the body reads `Model does not support audio modality` where the modality is refused and `The provided model identifier is invalid.` where `bedrock-runtime` does not carry the ID (section 5). An `audio_url` part is refused differently, with 400 `Failed to deserialize the JSON body into the target type: ?[0]: Invalid 'messages': Invalid 'content': value did not match any expected variant`, so the shape of the part is not what the first message is about (both models, both endpoints, `us-east-1`, 25 October 2026, 4 of 4 calls). Neither primary source says where audio is taken: both cards tick Speech and tick In-Region for each of their sixteen Regions, and `GetFoundationModel` reported `['SPEECH', 'TEXT']` in `us-east-1` on 20 October 2026. These models are Legacy, so plan the move off them that the top of this notebook sets out |


## Next

- General-purpose Mistral models and the size ladder:
  [`01-mistral-text-and-sizes`](01-mistral-text-and-sizes.ipynb)
- Another coding family: [`04-qwen/02`](../04-qwen/02-qwen3-coder-and-vision.ipynb)
  (Qwen3 Coder)
- Choosing a model and API across families:
  [`99-cross-cutting/01`](../99-cross-cutting/01-choosing-a-model-and-api.ipynb)
- Production hardening:
  [`99-cross-cutting/03`](../99-cross-cutting/03-production-hardening.ipynb)